# Core Generative AI — LLMs explained with code, charts & STAR 🧠

*A from-zero-to-interview workbook on **Generative AI** and **Large Language Models**. Every
concept comes with a **plain-English explanation**, a **working code snippet**, and where it
helps a **chart** — framed in the **Situation → Task → Action → Result (STAR)** template plus
**Problem → Solution → Alternatives → Drawbacks** so you can *defend* your choices.*

> **What you will be able to do after this notebook**
> - Explain how a transformer "reads" text (tokens → embeddings → attention).
> - Explain how an LLM picks the next word (softmax over a vocabulary, tuned by temperature/top-k).
> - Compare encoder-only vs decoder-only LLMs and when to fine-tune vs prompt vs use RAG.
> - Walk a real enterprise use case end-to-end (STAR) and pick the right technique.
> - Answer the classic traps ("why does it hallucinate?", "what makes it 'just' predicting?").

> **Runs entirely offline**, with only libraries already in the `qf` env
> (scikit-learn, pandas, numpy, matplotlib). No API keys, no model downloads — we build tiny
> self-contained demos that illustrate the *real* mechanics inside a big LLM.


## Before we start — the two templates you will reuse everywhere

**⭐ STAR — how you *describe* any GenAI task in an interview:**
| Field | What you say |
|---|---|
| **S**ituation | A real business/data problem (e.g. "support invoices pile up in a shared inbox"). |
| **T**ask | The GenAI formulation (summarise/classify/answer/generate). |
| **A**ction | The concrete pipeline: preprocessing → model → guardrails → evaluation. |
| **R**esult | **Numeric success criteria** — how do we *know* it worked? |

**🧭 P-S-A-D — how you *defend* an architectural choice:**
- **Problem** — why a naive / brute-force approach isn't enough.
- **Solution** — the technique you pick (prompting, fine-tuning, RAG, PEFT…).
- **Alternatives** — what else you could have used, and *when* each is better.
- **Drawbacks** — the honest weaknesses interviewers probe.

Most sections below are structured this way. Keep the two templates in your head the whole time.


## 1. What *is* Generative AI? Where do LLMs fit? 🌳

**One sentence:** Generative AI is the branch of machine learning that **creates new, plausible
content** (text, images, code, audio) instead of only mapping inputs to labels. An **LLM** is
a generative model *specifically for text* — it reads a prompt and produces a continuation.

**The hierarchy you must be able to draw:**
```
Artificial Intelligence        (machines mimicking intelligent behaviour)
 └─ Machine Learning            (learning patterns from data)
     └─ Deep Learning           (multi-layer neural networks)
         └─ Generative AI       (models that *produce* new content)
             └─ LLMs            (GPT-style models: text in → text out)
```

⭐ **STAR** — *Why would we even use GenAI?*
- **S**ituation: teams have unstructured text (emails, tickets, contracts, logs) and need to
  produce natural answers, drafts or structured data from it.
- **T**ask: generate plausible, context-relevant *new* text (answer a question, summarise,
  rewrite, extract).
- **A**ction: pick an LLM; decide **prompt + retrieval (RAG)** vs **fine-tuning (PEFT)**; add
  guardrails and evaluation.
- **R**esult: measurable lift in task accuracy / % of drafts used / reduced handling time.

🧭 **P-S-A-D** — *Why not just regular ML?*
- **Problem**: classical ML (logistic regression, SVMs on BoW/TF-IDF) can't produce fluent new
  prose or handle a *novel, unseen* phrasing.
- **Solution**: transformer-based generative models trained on vast text learn flexible,
  long-range language patterns.
- **Alternatives**: for *fixed* tasks (spam detection, sentiment) a cheap TF-IDF + linear model
  is often better and 100x cheaper — GenAI is not always the answer.
- **Drawbacks**: cost, latency, hallucination risk, and — for narrow structured tasks —
  overkill. Choose by **task determinism** (Section 12).


In [ ]:
# 1a. Setup + a map of the AI family tree (rendered as a chart)
import numpy as np, pandas as pd, math, re
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA
plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.figsize'] = (6.5, 3.2)

levels = [
 ("Artificial Intelligence", "all machine intelligence", "#0f3b5f"),
 ("Machine Learning", "learn rules from data", "#2a6bb0"),
 ("Deep Learning", "neural nets with many layers", "#4c9ad6"),
 ("Generative AI", "creates NEW content", "#7cb8e4"),
 ("Large Language Models (GPT)", "reads tokens, writes tokens", "#cfe4f5"),
]
fig, ax = plt.subplots()
y = np.arange(len(levels))[::-1] + 0.72
for yy, (label, sub, col) in zip(y, levels):
    ax.barh(yy, 1, color=col, height=0.5)
    ax.text(0.015, yy, f"{label}  —  {sub}", va='center', ha='left',
            color='#0b2b45', fontsize=9)
ax.axis('off'); ax.set_xlim(0, 1.3); 
ax.set_title("Generative AI sits *inside* deep learning; LLMs are the text flavour", fontsize=10)
plt.tight_layout(); plt.show()


## 2. Tokenisation — how text enters the model ✂️

**Key idea:** an LLM doesn't read characters or whole words. It splits text into **tokens**
— roughly sub-word units. A word like *unhappiness* becomes `un` + `happiness`; common
tokens are whole words (`the`, `of`); rare ones get split more. The model predicts the *next
token*.

**Why sub-words and not words?**
- **Vocabulary explosion** — true word-level vocabularies are millions of entries; sub-words
  give you `~50k` tokens that cover almost any word by composition.
- **Out-of-vocabulary (OOV)** — any new/misspelt word can be decomposed into known pieces
  (e.g. `gpt-rag` → `gpt` + `-` + `rag`).
- Tokenisers are usually **learned with byte-pair encoding (BPE)**: repeatedly merge the two
  most frequent adjacent units into one token.

> 💡 **Interview soundbite:** "The context window is counted in *tokens*, not words — ~1.3
> tokens per English word; 8k context ≈ 6k words. That's why paying attention to token count
> decides cost and whether the prompt fits." 


In [ ]:
# 2a. Word-level tokenisation + a BPE-style sub-word intuition
sentence = "The quick brown fox jumps over the lazy dog"
tokens = re.findall(r"\b\w+\b", sentence.lower())
print("Sentence     :", sentence)
print("Word tokens  :", tokens, "->", len(tokens), "tokens")
sub = sentence.lower().replace("fox", "fo" + " " + "kz")  # simulate an OOV split
print("With an OOV  : 'fox' could become", "'fo' + 'kz'", "if 'kz' is in the vocab")

# Every token costs context — visualise per-token length
cnt = pd.Series([len(t) for t in tokens], index=tokens)
ax = cnt.plot.bar(color="#7cb8e4")
ax.set_title("Tokens are sub-word units — lengths vary")
plt.xticks(rotation=45, ha='right'); plt.ylim(0, 6)
plt.tight_layout(); plt.show()


## 3. Embeddings — turning tokens into vectors the model can reason about 🧮

After tokenisation each token is mapped to a **dense vector (embedding)** — e.g. 768 or 4096
dimensions. Embeddings are **learned during training** so that *similar-in-meaning* words land
close together in vector space:

`dog ≈ puppy`, `cat ≈ kitten`, but `banana` is far away.

**Why vectors?** Once words are numbers, we can do maths — cosine similarity measures
closeness, and (famously) `king − man + woman ≈ queen`.

⭐ **STAR** — embeddings at work
- **S**: search/proximity problems over text (semantic search, clustering, dedup).
- **T**: represent every text as a vector so "meaning distance" ≈ vector distance.
- **A**: embed with TF-IDF or a transformer encoder; compare with cosine similarity.
- **R**: retrieval recall, cluster purity — numeric and measurable.

🧭 **P-S-A-D** — *Why embeddings instead of one-hot?*
- **Problem**: one-hot vectors are huge (vocab-size), sparse, and treat every word as unrelated.
- **Solution**: small *dense* embeddings, trained so similar words are near each other.
- **Alternatives**: pre-trained encoder models (BERT-style) for *contextual* embeddings;
  TF-IDF if you need interpretability.
- **Drawbacks**: embedding axes aren't human-readable; a meaning nuance can collapse them.


In [ ]:
# 3a. A tiny hand-made embedding space + cosine similarity (clustering demo)
words  = ["cat","dog","kitten","puppy","car","truck","buy","sell"]
emb = np.array([            # 3-dim toy embeddings, hand-crafted to cluster by meaning
 [ .90, .10, .20],   # cat
 [ .85, .15, .10],   # dog
 [ .70, .30, .20],   # kitten
 [ .65, .35, .15],   # puppy
 [ .20, .90, .10],   # car
 [ .18, .85, .20],   # truck
 [ .10, .20, .90],   # buy
 [ .15, .10, .85]])  # sell
low = PCA(n_components=2).fit_transform(emb)
plt.scatter(low[:,0], low[:,1], s=140, c="#4c9ad6")
for (x,y),w in zip(low, words): plt.annotate(w, (x,y), fontsize=10)
plt.title("Tiny embeddings in 2D (PCA) — similar meanings cluster together")
plt.gca().set_axis_off(); plt.tight_layout(); plt.show()

print("cosine(cat, kitten) =", round(cosine_similarity([emb[0]],[emb[2]])[0,0], 3))
print("cosine(cat, truck)  =", round(cosine_similarity([emb[0]],[emb[4]])[0,0], 3))
print("-> animals cluster; vehicles are far from the cat.")


## 4. Attention & self-attention — the heart of transformers ❤️

**The core problem old models had:** sequential (RNN/GRU) models read word-by-word and
*forgot* long-range links ("the monkey that lived in the zoo ... **it** ate"). 

**The transformer fix — `Self-Attention`:** while reading a word, the model computes how much
*every other word* should influence it, using three learnable projections:

```
Q = X·Wq     "what am I looking for?"
K = X·Wk     "what do I offer?"
V = X·Wv     "what do I actually contribute?"
score(Q,K) = softmax( (Q·Kᵀ) / √d )     ← attention weights
attention  = softmax_weights · V         ← weighted combination = new token vector
```

- `Q·Kᵀ` measures relevance between each pair of tokens.
- Dividing by **√d** stops the dot products from exploding (numerical stability).
- **softmax** turns raw scores into a *probability* over the keys.
- The result is a **contextual embedding** — each token now "knows about" its neighbours.

> 💡 **Soundbite:** "Attention lets every token attend to *every* token — the whole sequence
> in parallel — because the cloud/key mechanism is an arbitrary lookup, not a fixed window.
> That's **O(n²)** in tokens, but GPUs parallelise it; it's why transformers beat RNNs."


In [ ]:
# 4a. Self-attention from first principles on a 4-token sentence
tokens = ["I", "love", "chocolate", "cake"]
X = np.array([[.9,.3],    # I
              [.6,.8],    # love
              [1.0,.7],   # chocolate
              [.5,.9]])   # cake          (2-dim embeddings, to keep it readable)

# For a toy demo we treat Wq and Wk as identity: Q=X, K=X, V=X
Q, K, V = X.copy(), X.copy(), X.copy()
raw = Q @ K.T / math.sqrt(K.shape[1])        # relevance scores, scaled
att = np.exp(raw); att = att / att.sum(axis=1, keepdims=True)   # softmax over the row
ctx = att @ V                                 # weighted combination -> context vectors

plt.imshow(att, cmap="Blues")
plt.colorbar(label="attention weight")
plt.xticks(range(4), tokens); plt.yticks(range(4), tokens)
plt.xlabel("Key  (which tokens are relevant)"); plt.ylabel("Query  (the token being read)")
plt.title("1 self-attention head: each output is a blend of all tokens")
plt.tight_layout(); plt.show()

print("Attention matrix (rows sum to 1):"); print(att.round(3))
print("Context vector for 'love':", ctx[1].round(3))


## 5. The transformer block & the three LLM "flavours" 🏗️

**A "transformer block"** (stacked ~12→96 times) does roughly:
1. **Self-attention** — tokens exchange information (Section 4).
2. **Feed-forward MLP** — non-linear per-token transformation.
3. **Layer-norm + residual connections** — stable deep training.
4. **Positional encoding** — because attention is order-agnostic, we inject position info.

**Three architecture families — memorise the differences:**
| Flavour | Masking | Typical use | Example |
|---|---|---|---|
| **Encoder-only** | *bidirectional* (sees left & right) | understand/classify: NER, sentiment, embeddings | BERT |
| **Decoder-only** | *causal* (only past tokens) | **generate**: the LLM next-token writers | GPT, Claude, Llama |
| **Encoder–Decoder** | both | transformation: translation, summarisation | T5, BART |

> 💡 **Big-LLM point:** modern chat LLMs are **decoder-only**: they literally *predict the
> next token* (prefix + the tokens they generated so far), then feed their own output back in —
> that's what "autoregressive" means. BERT *reads*; GPT *writes*.

⭐ **STAR** — picking the right architecture
- **S**: a finance team wants (a) to tag emails and (b) to draft replies.
- **T**: (a) classification → **encoder**; (b) generation → **decoder**.
- **A**: use a PEFT fine-tuned BERT for tagging; a decoder LLM (prompted) for drafting.
- **R**: accuracy on tags + % replies used, both measured.


## 6. How an LLM actually *generates*: next-token + decoding 🎰

At every step the model outputs a **logit (score) per token** in the vocabulary. Those logits
are turned into a probability distribution via **softmax**, and the next token is sampled from
it. This one-step loop runs until the model emits a stop token:

```
prompt:  "The capital of France is"
logits:  ... Paris: 9.1   London: 5.0   Berlin: 4.2   banana: -2.0 ...
sample:  -> "Paris"        (the most likely — but not always the chosen one!)
```

We rarely pick the raw argmax. Three knobs control the shape of the sampling distribution:

| Knob | What it does |
|---|---|
| **temperature** `T` | scale logits `÷ T` before softmax. `T<1` sharpens → more predictable; `T>1` flattens → more diverse/creative. |
| **top-k** | keep only the `k` most likely tokens, drop the rest (their probability = 0). |
| **top-p (nucleus)** | keep the smallest set of tokens whose *cumulative* probability ≥ p. Adaptive version of top-k. |

> 💡 **Soundbite:** "Lower temperature = more deterministic but repetitive; higher = creative
> but riskier. For *structured* tasks (JSON, code) you want low T; for creative copy you raise
> it. top-k and top-p trim the unlikely tail so you don't sample gibberish."


In [ ]:
# 6a. Temperature softmax — see the distribution flatten / sharpen
logits = np.array([3.2, 2.4, 1.1, 0.5, -1.2])
def softmax(z):
    e = np.exp(z - z.max()); return e / e.sum()
fig, ax = plt.subplots()
x = np.arange(len(logits))
for T in [0.2, 1.0, 5.0]:
    ax.plot(x, softmax(logits / T), marker='o', label=f"T = {T}")
ax.set_xticks(x); ax.set_xticklabels(["token A","B","C","D","E"])
ax.set_xlabel("vocabulary token"); ax.set_ylabel("probability")
ax.legend(); ax.set_title("Temperature controls the sharpness of the next-token distribution")
plt.tight_layout(); plt.show()


In [ ]:
# 6b. A tiny offline "language model": P(next | previous) from a corpus
# A real LLM conditions on the ENTIRE history via attention; this toy only conditions
# on the previous word (a bigram model) — enough to show next-token generation mechanics.
from collections import defaultdict, Counter
corpus = "the cat sat on the mat the dog ran through the park the cat chased the dog".split()
trans = defaultdict(Counter)
for a, b in zip(corpus, corpus[1:]):
    trans[a][b] += 1

def next_word(w, rng):
    opts = list(trans[w])
    if not opts: return "the"
    probs = [trans[w][o] / sum(trans[w].values()) for o in opts]
    return rng.choice(opts, p=probs)

rng = np.random.default_rng(1)
generated = ["the"]
for _ in range(9):
    generated.append(next_word(generated[-1], rng))
print("Bigram 'model' says:", " ".join(generated))
print("This is next-token-prediction, just with a 1-word memory instead of a huge context.")


## 7. The context window — your prompt's budget 📐

The **context window** is the max number of tokens the model attends to per request
(4k / 8k / 128k / 1M are common). Everything the model sees — system prompt, user prompt,
documents, earlier conversation — shares this budget.

**Practical arithmetic (memorise):**
- *Tokens ≈ 0.75–0.8 × characters* (English), ≈ 1.3 tokens per word.
- If documents are long, sending the whole text may exceed the window → you hit truncation
  or cost blow-up → that's when **RAG + chunking** (Section 10) beats "stuff everything in".
- Each generated output token also counts — long answers use budget and cost.

> 💡 **Soundbite:** "Cost ≈ (input tokens + output tokens) × price; the context window is the
> hard ceiling. That's why we implement retrieval + chunking for big corpora."


## 8. How the model *becomes* useful: pretrain → instruct → align 🔁

Reading an LLM "paper trail" requires knowing what shape the data / loss took:

**1. Pre-training (unsupervised, the expensive step).** Learn the language itself from
terabytes of text by predicting tokens:
- **Causal LM (LLMs):** given tokens `[t₁..tₙ]`, predict `tₙ₊₁` (autoregressive — a "read-left" mask).
- **Masked LM (BERT):** hide random tokens, predict the blanks using **both** sides.

**2. Supervised fine-tuning (SFT / instruction tuning).** Feed (instruction, ideal answer)
pairs so the base "next-token machine" learns to *follow directions* (turns it into an
assistant).

**3. Alignment (RLHF).** Use a reward model on human preferences to nudge outputs toward
"harmless + helpful". This is why chat models refuse/format better than raw base models.

**4. PEFT (parameter-efficient fine-tuning).** For *your* domain (e.g. legal drafting) you
don't retrain 100B params — freeze the base and train a tiny adapter / LoRA low-rank update
(maybe 1–2% of params). Cheap, fast, keeps the backbone.

> 💡 **Soundbite — the four stages:** *pre-train* (language), *instruct* (follow orders),
> *align/RLHF* (behave), *fine-tune/PEFT* (your domain). Most teams *never* pre-train; they
> prompt, or PEFT an existing model.


## 9. Prompting techniques you must name: ICL, few-shot, CoT 🪄

- **Zero-shot**: no example — "Classify this email: …". Works surprisingly well on modern LLMs.
- **Few-shot (in-context learning, ICL)**: give 3–5 labelled examples *inside the prompt* to
  teach format/behaviour without any training update.
- **Chain-of-thought (CoT)**: ask the model to "think step by step" → dramatically better maths
  & reasoning; the processing is all in-prompt.
- **Structured outputs**: ask for JSON with a schema, or constrain sampling to valid tokens.

🧭 **P-S-A-D — why few-shot and not fine-tuning?**
- **Problem**: you have a new output format and zero labelled data.
- **Solution**: few-shot examples in the prompt teach the format instantly.
- **Alternatives**: PEFT/LoRA if you have hundreds of examples and want lower cost per call;
  a classical classifier if the task is fixed and cheap.
- **Drawbacks**: examples eat context; too few/misleading examples can *hurt*; large prompts
  cost more. Fine-tune when you outgrow the prompt.


In [ ]:
# 9a. Few-shot "classification" offline: nearest-neighbour over TF-IDF examples
# (A stand-in for learning the format from prompt examples — uses similarity, not training.)
few_shot = [
 ("Declare invoice 4482",        "finance"),
 ("The payment has cleared",     "finance"),
 ("Bug: API returns HTTP 500",   "engineering"),
 ("Deploy the new image now",    "engineering"),
 ("Approve the quarterly budget","finance"),
 ("Refactor the auth module",    "engineering"),
]
test = ["The invoice payment failed", "Deploy the API image now"]
v = TfidfVectorizer(token_pattern=r"\w+")
M = v.fit_transform([t for t, _ in few_shot] + test)
sims = cosine_similarity(M[:len(few_shot)], M[len(few_shot):])
for j, t in enumerate(test):
    i = int(sims[:, j].argmax())
    print(f"{t!r}  ->  {few_shot[i][1]}   (sim to closest exemplar = {sims[i, j]:.2f})")


## 10. RAG — Retrieval-Augmented Generation (talking about *your* data) 🔎

**The problem with a raw LLM:** it only knows what was in its training corpus; it can't know
*your* internal documents and it can *invent* facts (hallucinate) when asked something specific.

**RAG = retrieve first, then generate.**
```
User question ──► retrieve top-k relevant doc chunks (embeddings/TF-IDF search)
                     │
                     ▼
        prompt = question + [retrieved chunks]  ──►  LLM answers *grounded* in the chunks
```
- **Retriever**: embed every doc chunk into a vector store; at query time find the closest by
  cosine similarity; the top-k chunks become context.
- **Why it's the #1 enterprise pattern**: answers are *grounded* in current, private sources,
  hallucinations drop, context stays small, and you don't retrain.
- **Chunking**: split long docs into ~500–1000 token chunks with overlap so the right text
  actually lands inside the window.

> 💡 **Soundbite:** "RAG says *bring the facts to the model* instead of *teach the model your
> facts*. It's the cheapest, most auditable way to make an LLM answer over private documents —
> combine it with a low enough temperature and cite the source chunk."


In [ ]:
# 10a. A mini RAG retriever: TF-IDF document store + top-k answers (offline)
docs = {
 "policy": "Refunds are processed within 10 business days to the original payment method.",
 "itil":   "Onboarding tickets are assigned to L1 support and escalated after 24 hours.",
 "pay":    "Wire transfers over 50k require dual approval from finance and treasury.",
 "hr":     "Annual leave must be requested two weeks in advance through the HR portal.",
}
queries = ["How do refunds work for my payment?", "Wire transfers over 50k need approval"]

v = TfidfVectorizer(token_pattern=r"\w+")
names = list(docs)
M = v.fit_transform(list(docs.values()) + queries)
S = cosine_similarity(M[:len(docs)], M[len(docs):])          # docs x queries

for qi, q in enumerate(queries):
    hits = sorted(range(len(docs)), key=lambda k: -S[k, qi])[:2]
    print(f"Q: {q!r}")
    for h in hits:
        print(f"   -> hit '{names[h]}' (score {S[h, qi]:.2f}): {docs[names[h]]}")
print("Note: lexical TF-IDF needs shared words - a paraphrase like 'get my money back' scores ~0. That is why production RAG uses semantic embeddings, not raw word overlap.")


## 11. Hallucination & how we *evaluate* generative text 🩺

**Hallucination** = the model outputs plausible but *untrue* text (it is always "just"
auto-completing — it can't tell truth from fluent-sounding). Mitigations: RAG grounding,
temperature < 1, self-checking, and *citing sources*.

**Evaluation metrics you should know:**
| Metric | Measures | Comment |
|---|---|---|
| **Perplexity** | how "surprised" a model is by a text | `exp(−mean log P(token))`; lower = more confident. Intrinsic, cheap. |
| **BLEU** | n-gram overlap vs a reference (translation) | precision-oriented; good for translation. |
| **ROUGE** | n-gram/overlap recall (summarisation) | F-score of overlap with a reference summary. |
| **Factual / grounded** | % answers supported by a source | human or LLM-judge; most important in RAG. |

> 💡 **Soundbite:** "Perplexity tells you a *model's* confidence, not *truth*. For a product we
> evaluate **downstream**: did the answer match the content (grounded), is it helpful (human) —
> never trust a single auto-metric as the whole story."


In [ ]:
# 11a. Perplexity worked by hand (a model that assigns each true token a probability)
probs = np.array([0.9, 0.6, 0.3, 0.95, 0.8])     # P(correct token) the model gave each position
perp = math.exp(-np.mean(np.log(probs)))
another = probs * 0.7                            # a "worse" model: less confident on EVERY token
print("Perplexity (confident model):", round(perp, 3))
print("Perplexity (worse model)    :", round(math.exp(-np.mean(np.log(another))), 3),
      "  (higher = worse)")
pd.DataFrame({"token_prob": probs, "per_token_loss": -np.log(probs)}).plot.bar(color="#7cb8e4")
plt.title("Perplexity = average per-token surprise, exponentiated")
plt.ylabel("loss for that token"); plt.tight_layout(); plt.show()


## 12. Gen AI use cases — deciding *when* to use it 🎯

**The golden rule: match the task's determinism to the technique.** Put each use case on a
*creativity ⇄ determinism* spectrum:
```
creative / open-ended        ────────────►        deterministic / exact
copywriting, brainstorming   summarise, code     classification, extraction,
                             translation, RAG    inference, structured fields
```

| Use case | GenAI role | STAR "R" (how do we know it works?) |
|---|---|---|
| **Chatbot / support** | draft answers grounded in FAQ (RAG) | first-contact resolution %, CSAT |
| **Content & copy** | rewrite / draft for tone | % drafts used, time-to-draft |
| **Code generation** | scaffold, explain, convert | % compiled, PR accepted |
| **Summarisation** | long docs → TL;DR | ROUGE + human "captures key points" |
| **Document Q&A** | RAG over policies/contracts | answer grounded in source, 0 found-but-wrong |
| **AI extraction** | NER / structured fields from emails | field accuracy, linkage to systems (see the NLP notebook case) |
| **Classification / risk** | label, triage | precision/recall — often a **classical** model wins on cost |

> 💡 **Soundbite:** "Don't reach for a 175B-param generative model for *structured extraction*
> where a tiny encoder or even TF-IDF logistic regression is cheaper and more reliable. GenAI
> shines when the task needs *fluent, novel, context-dependent text*; classical ML shines when
> the output is a fixed label."


In [ ]:
# 12a. Map use cases onto the creativity ⇄ determinism spectrum
cases = {
 "Copywriting": (.92,.92),  "Chatbots": (.72,1.0),  "Brainstorming": (.95,.8),
 "Code generation": (.55,.8), "Summarisation": (.42,.5), "Document Q&A (RAG)": (.4,.35),
 "Translation": (.32,.4),  "AI extraction / NER": (.12,.2), "Classification": (.06,.1),
}
fig, ax = plt.subplots()
for name,(cx,cy) in cases.items():
    ax.scatter(cx, cy, s=160, color="#4c9ad6"); ax.annotate(name, (cx, cy), fontsize=8)
ax.set_xlabel("open-ended creativity  →"); ax.set_ylabel("exactness / determinism  →")
ax.set_title("Use GenAI where it's open-ended; use classical ML where it's fixed")
ax.set_xlim(0,1); ax.set_ylim(0,1.05)
plt.tight_layout(); plt.show()


## 13. GenAI vs classical ML — the decision framework ⚖️

Ask three questions in order:
1. **Is the output a fixed label / number?** → classical ML (or a tiny encoder), not a big LLM.
2. **Do we need fluent, novel, context-aware text?** → GenAI (LLM).
3. **Does the answer depend on private/current facts?** → GenAI **+ RAG** (retrieve the facts).

| Decision | Choose |
|---|---|
| Spam / sentiment / fraud flag | TF-IDF + logistic regression, or BERT encoder |
| Extract structured fields from emails | NER / regex / small encoder (see NLP notebook) |
| Draft a reply, summarise a report, answer over docs | LLM + RAG |
| Translate / paraphrase | Encoder–decoder (T5) or LLM |
| Explain *why* a model decided | classical ML (feature importances) — LLMs are hard to interpret |

🧭 **P-S-A-D — "why not just always use the biggest LLM?"**
- **Problem**: cost, latency, hallucination, and non-determinism hurt *fixed* tasks.
- **Solution**: pick the *smallest tool that meets the task*; escalate to GenAI only when the
  task is genuinely generative.
- **Alternatives**: hybrid — classical rules/classifier for triage, LLM for the free-text part.
- **Drawbacks**: hybrid adds moving parts; GenAI-only is simpler but pricier and riskier.


## 14. Cheat-sheet & the classic interview traps 🪤

**10 numbers/names worth memorising**
- Tokens ≈ 1.3 per word; ≈ 0.75 × characters → budget with the context window.
- Self-attention is **O(n²)** in context length (why 128k+ contexts need sparse/sliding tricks).
- Three projections: **Q** (query), **K** (key), **V** (value); score = `softmax(Q·Kᵀ/√d)`.
- Encoder = bidirectional (BERT, reads); decoder = causal (GPT, writes); both = T5/BART.
- Pipeline: **pre-train → instruct (SFT) → align (RLHF) → PEFT/LoRA**.
- Sampling knobs: **temperature**, **top-k**, **top-p/nucleus**.
- RAG = retrieve top-k chunks → prompt → grounded answer (kills hallucination, no retrain).
- Eval: **perplexity** (intrinsic), **BLEU/ROUGE** (reference overlap), **grounded/factual** (product).
- PEFT/LoRA fine-tunes ~1–2% of params — freezes the giant backbone.
- Hallucination mitigation: **RAG + low temperature + cite sources + self-check**.

**Traps they will throw at you — scripted answers**
1. *"Is the LLM just predicting the next word?"* — Yes, autoregressively, but the *conditioning*
   (huge context, attention, instruction data, RLHF) is what makes it useful. "Predicting the
   next token" is the *mechanism*; not a dismissal of the model.
2. *"Why does it hallucinate?"* — It always maximises *plausibility*, not *factuality*; it had
   no access to your data and can't "check". Solution: RAG + temperature + citations.
3. *"Why not always the biggest model?"* — Cost/latency per token, and overkill for fixed labels
   where classical ML wins. Match the tool to task determinism.
4. *"How do you evaluate it?"* — Never one number. Cross-Entropy/perplexity + task metrics
   (BLEU/ROUGE/accuracy) + human/grounded checks on a labelled slice.
5. *"Tokens vs words?"* — Sub-word units; standards are BPE; rule of thumb 1.3 tokens/word.
6. *"Prompting vs fine-tuning?"* — Prompting = no training, flexible, but eats context & cost.
   Fine-tune (PEFT) = when you have examples, need speed/consistency at scale.
7. *"RAG vs fine-tuning for knowledge?"* — RAG for *facts/knowledge* (updatable, grounded);
   fine-tuning for *style/format/behaviour*. Retrieval answers; fine-tuning styles.

**A 60-second STAR "big answer" — *Answer over internal docs***
> Schat / **S**: finance ops need instant, grounded answers over thousands of policy PDFs.
> **T**: a RAG assistant that retrieves the relevant clause and answers, citing it.
> **A**: chunk docs (~500 tokens, overlap), embed to a vector store, at query time retrieve top-k
> by cosine similarity, prompt the LLM with question + chunks + "answer only from the context",
> temperature 0.2.
> **R**: ~90% of answers grounded in a source (LLM-judge), retrieval recall ≥ 0.85, CSAT up —
> and 0 successful-rejection of a wrong-fact answer.

**_The honest line to end any GenAI interview:_** "A big model is a fluent guesser. My job is to
give it the right context, the right shape of prompt, the right temperature — and to measure
whether the *product* outcome really improved."
